# Sentiment Analysis 
###  ML / NLP on real-world text data

**Goal:** Classify customer tweets directed at airlines as Positive, Neutral, or
Negative, using classical NLP (TF-IDF) and classical ML models. Framed as a **complaint triage system**: the real business
use case is routing negative, high-urgency messages to support teams faster.

**Dataset:** Twitter US Airline Sentiment (Kaggle)
https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment
File: `Tweets.csv` — ~14,600 tweets, labeled Positive / Neutral / Negative,
directed at 6 major US airlines. Place it in the same folder as this notebook.

**Concepts covered:** text preprocessing (cleaning, tokenization, stopword
removal), TF-IDF vectorization, multi-class classification, model comparison
(Logistic Regression, Multinomial Naive Bayes, Linear SVM), and business
framing around complaint triage.


## 1. Imports & Setup

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


## 2. Load & Explore the Data

As with any classification project, we check class balance first. Airline
sentiment data is typically skewed toward negative tweets — people tweet at
airlines far more often to complain than to praise — which itself is a
realistic, business-relevant imbalance worth calling out.


In [ ]:
df = pd.read_csv("Tweets.csv")
print(df.shape)
df[['airline', 'airline_sentiment', 'text']].head()


In [ ]:
print(df['airline_sentiment'].value_counts())
print(df['airline_sentiment'].value_counts(normalize=True).round(3))

sns.countplot(x='airline_sentiment', data=df, order=['negative', 'neutral', 'positive'])
plt.title('Sentiment Class Distribution')
plt.show()


In [ ]:
sns.countplot(x='airline', hue='airline_sentiment', data=df,
              hue_order=['negative', 'neutral', 'positive'])
plt.title('Sentiment by Airline')
plt.xticks(rotation=30)
plt.legend(title='Sentiment')
plt.show()


## 3. Text Preprocessing

Tweets need specific cleaning: removing @mentions (e.g. "@united"), URLs,
hashtags' punctuation, and non-alphabetic characters, then lowercasing and
stripping stopwords. We deliberately keep this simple and transparent rather
than using a heavier pipeline, since interpretability of the pipeline itself
matters for this kind of project.


In [ ]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'@\w+', '', text)              # remove @mentions
    text = re.sub(r'http\S+|www\S+', '', text)     # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)           # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()        # collapse whitespace
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_tweet)
df[['text', 'clean_text']].sample(5, random_state=RANDOM_STATE)


In [ ]:
# Sanity check: make sure cleaning didn't empty out too many tweets
empty_after_cleaning = (df['clean_text'].str.len() == 0).sum()
print(f"Tweets that became empty after cleaning: {empty_after_cleaning}")

df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print("Remaining rows:", len(df))


## 4. Quick EDA on Cleaned Text — Most Common Words by Sentiment

A simple, useful sanity check before modeling: do the most frequent words per
class actually make sense? This also doubles as a lightweight, interpretable
"what's driving this sentiment" view, similar in spirit to the SHAP work in
the credit risk project but far cheaper to compute.


In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, sentiment in zip(axes, ['negative', 'neutral', 'positive']):
    words = ' '.join(df[df['airline_sentiment'] == sentiment]['clean_text']).split()
    top_words = Counter(words).most_common(15)
    labels, counts = zip(*top_words)
    ax.barh(labels[::-1], counts[::-1])
    ax.set_title(f'Top words — {sentiment}')

plt.tight_layout()
plt.show()


## 5. Train/Test Split & TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) weighs words by how
distinctive they are to a document relative to the whole corpus — common
words like "flight" get down-weighted, while more sentiment-specific words
get more weight. We fit the vectorizer only on the training set to avoid
leaking test-set vocabulary statistics into training.


In [ ]:
X = df['clean_text']
y = df['airline_sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train shape:", X_train_tfidf.shape, " Test shape:", X_test_tfidf.shape)


## 6. Model Comparison — Logistic Regression vs. Naive Bayes vs. Linear SVM

Same "honest comparison" approach as the other projects: train several
classical models on the identical features and compare, rather than assuming
one algorithm is automatically best.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(class_weight='balanced', random_state=RANDOM_STATE, max_iter=5000),
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    predictions[name] = preds
    results[name] = {
        'accuracy': accuracy_score(y_test, preds),
        'macro_f1': f1_score(y_test, preds, average='macro'),
        'weighted_f1': f1_score(y_test, preds, average='weighted')
    }
    print(f"=== {name} ===")
    print(classification_report(y_test, preds))
    print()


In [ ]:
results_df = pd.DataFrame(results).T
print(results_df.round(3))

results_df.plot(kind='bar', figsize=(10, 5), ylim=(0, 1))
plt.title('Model Comparison — Accuracy, Macro-F1, Weighted-F1')
plt.xticks(rotation=0)
plt.show()


## 7. Confusion Matrices — Where Do Models Actually Get Confused?

Macro-F1 punishes poor performance on the minority classes (neutral, positive)
much more than accuracy does, since accuracy alone would be dominated by the
majority negative class. The confusion matrix shows exactly which sentiment
pairs get mixed up — typically neutral vs. positive/negative boundary cases,
which makes intuitive sense since sarcasm and mild complaints are genuinely
hard to classify correctly.


In [ ]:
best_model_name = results_df['macro_f1'].idxmax()
print(f"Best model by macro-F1: {best_model_name}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels_order = ['negative', 'neutral', 'positive']
for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds, labels=labels_order)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels_order, yticklabels=labels_order)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()


## 8. Most Influential Words per Class (Logistic Regression Coefficients)

For the linear models, we can directly inspect which words push a prediction
toward each sentiment class — a simple, fast form of interpretability, well
suited to a business audience without needing SHAP or other heavier tooling.


In [ ]:
log_reg = models['Logistic Regression']
feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, class_label in zip(axes, log_reg.classes_):
    class_idx = list(log_reg.classes_).index(class_label)
    coefs = log_reg.coef_[class_idx]
    top_idx = np.argsort(coefs)[-15:]
    ax.barh(feature_names[top_idx], coefs[top_idx])
    ax.set_title(f'Top words pushing toward: {class_label}')

plt.tight_layout()
plt.show()


## 9. Business Framing — Complaint Triage Simulation

The real-world value of this model isn't the sentiment label itself — it's
using that label to **route** messages. We simulate a simple triage rule:
any tweet predicted negative gets flagged as "high priority" for the support
team, and we check what fraction of tweets that would route to the team.


In [ ]:
best_preds = predictions[best_model_name]
priority_flags = (best_preds == 'negative')

print(f"Model: {best_model_name}")
print(f"Tweets flagged as high-priority (negative): {priority_flags.sum()} out of {len(priority_flags)} "
      f"({priority_flags.mean():.1%})")

# How many actual negative tweets did we correctly catch? (recall on negative class)
actual_negative = (y_test == 'negative')
caught = (priority_flags & actual_negative.values).sum()
print(f"Actual negative tweets correctly caught for priority routing: {caught} / {actual_negative.sum()} "
      f"({caught / actual_negative.sum():.1%} recall)")


## 10. Conclusions & Discussion

**Key findings to state based on your actual run's numbers above:**

- Report which model won on **macro-F1** (not just accuracy) — macro-F1 is the
  fairer metric here since the neutral/positive classes are much smaller than
  negative, and accuracy alone would hide poor performance on them.
- Linear SVM and Logistic Regression typically outperform Naive Bayes on
  TF-IDF text data, since Naive Bayes' independence assumption between words
  is a stronger simplification than the linear models need to make — worth
  confirming this holds in your actual run and explaining why if so.
- The confusion matrix will very likely show most errors between
  neutral and positive/negative, not between positive and negative — sentiment
  polarity is usually more clearly signaled by strong words, while "neutral"
  is inherently the hardest, fuzziest class to define, both for the model and
  for the humans who originally labeled the data.
- The interpretability step (top words per class) is a strong, cheap
  substitute for SHAP on this kind of linear text model — it's worth
  highlighting in an interview as a deliberate simplicity choice, not a
  limitation: linear model coefficients are directly interpretable without
  needing an additional explainability library.


### Possible extensions
- Compare against a pretrained transformer (e.g. DistilBERT via HuggingFace)
  fine-tuned on the same data, as a "classical vs. modern NLP" comparison
- Add aspect-based sentiment (e.g. separately flagging "delay," "customer
  service," "baggage" complaints) instead of one overall sentiment label
- Deploy as a small FastAPI/Streamlit app where a support team could paste in
  a tweet/message and get back a priority flag + top contributing words
